# 01 — GEE Hello World + CHIRPS proof-of-loop

**Week 1 goal:** prove the whole **GEE → Drive → Colab** loop works on one variable.

Run top-to-bottom in Google Colab. You need a free Earth Engine account and an EE *project* id
(create one at https://code.earthengine.google.com → Cloud project).

In [ ]:
# 1. Install the few packages Colab doesn't ship with
!pip install -q geemap rioxarray

# 2. (optional) clone your repo so you can import the library
# !git clone https://github.com/<you>/botswana-drought-flood.git
# %cd botswana-drought-flood
import sys; sys.path.append('src')

In [ ]:
import ee, geemap
ee.Authenticate()                       # follow the link, paste the token
ee.Initialize(project='YOUR-EE-PROJECT') # <-- replace with your project id

## Show Botswana

In [ ]:
from botswana_ds.gee.export import botswana_geometry, monthly_chirps
from botswana_ds.grid import make_grid

g = make_grid(); print('grid shape (H, W):', g.shape)

aoi = botswana_geometry()
m = geemap.Map(center=[-22, 24], zoom=5)
m.addLayer(aoi, {'color': 'red'}, 'Botswana boundary')
m

## CHIRPS monthly rainfall — visualize one month

In [ ]:
chirps = monthly_chirps('2016-01-01', '2017-01-01')   # 2016 = a known El Nino drought year
jan = chirps.first()
vis = {'min': 0, 'max': 150, 'palette': ['white', 'lightblue', 'blue', 'darkblue']}
m.addLayer(jan, vis, 'CHIRPS Jan 2016 (mm)')
m

## Export one month to Google Drive (closes the loop)
After running, watch the **Tasks** tab at https://code.earthengine.google.com — the GeoTIFF
lands in your Drive folder `BotswanaDroughtFloodSet`.

In [ ]:
task = ee.batch.Export.image.toDrive(
    image=jan,
    description='chirps_2016_01',
    folder='BotswanaDroughtFloodSet',
    fileNamePrefix='chirps_2016_01',
    region=aoi, scale=5566, crs='EPSG:4326', maxPixels=1e10)
task.start()
print('Export started — check the GEE Tasks tab.')

## Sanity check (Week-1 success criteria)
- Botswana boundary renders.
- Rainfall map is mostly dry (low values) over the Kalahari, wetter in the north.
- A GeoTIFF appears in Drive.

Next: `02_export_pipeline.ipynb` — loop `export_variable_to_drive` over all drought-core variables.